# Computational Theory Assignment Problem (SHA-256)


## Problem 1: Representing SHA-256 Data
Before we go ahead with the implementation of SHA-256, I'll first clarify a few things, mainly how input, output, and intermediate data in Python.

NOTE: the small python example I give for each data representation is just a simplified example for demo's sake, the actual implementation is after that.

### 32-bit words
Unlike C/C++/Rust where there's explicit `uint32_t` typing, Python is dynamically typed and its regular integers are not fixed-width. SHA-256 requires every word to behave as an unsigned 32-bit value, so the actual implementation will use NumPy's `np.uint32` type.

Initially, I wanted to go with an `Int32` class wrapper to make it safer, but decided against it because of a few reasons.
1. Performance overhead - SHA-256 produces a 32-bit result in every single operation, and a custom class would create object allocation overhead in the thousands of operations in the function.
2. Lose out on clean bitwise op syntax - a custom class would need many overloaded operators (i.e. `__add__`, `__xor__`, `__rshift__`, etc.), which adds work and more places where things could go wrong.
3. KISS principle, this one is self explanatory

The small example above uses a native Python `int` with a 32-bit mask to demonstrate the idea. In the actual implementation, the reusable `to_uint32` helper applies the mask and converts the result to `np.uint32`, keeping every SHA-256 word within the required 32-bit range.

In [1]:
# this is just for demo, we'll be encapsulating the masking in a helper function later on
word = 0x12345678
word = word & 0xffffffff

print("word in decimal = ", word) # 305419896 (0x12345678 in decimal)
print("word as hexadecimal = ", f"{word:08x}")  # 12345678
# x means hex, 08 means 8 digit, pad if necessary, same format as in C printf

word in decimal =  305419896
word as hexadecimal =  12345678


### sequences of 32-bit words
For the actual implementation, I'll use a mutable NumPy array with `dtype=np.uint32` to represent a sequence of words. This keeps each element fixed at 32 bits and is convenient when the message schedule is mutated during SHA-256.

The earlier `list[int]` example is a simplified demonstration of a word sequence. A Python list is easy to understand, but it does not enforce the 32-bit unsigned representation. The `to_uint32_array` helper below creates the NumPy representation. `bytes` and `bytearray` are alternatives for raw byte data, but word operations would require repeated conversion between bytes and integers.

In [2]:
words = [0x12345678, 0x90abcdef, 0x00000001] # list of 3 words as a demo

print("len(words) = ", len(words))
print([f"{word:08x}" for word in words])

len(words) =  3
['12345678', '90abcdef', '00000001']


### input messages
For the input messages we'll use python's immutable `bytes` type because sha256 operates at a byte level, the `bytes` type is evidently byte-level precise, I'm not going to be using characters because UTF-8 strings can range from 1-4 bytes long e.g. ඞ (0xE0 0xB6 0x9E) is 3 bytes long meanwhile the char `a` is 1 byte long. 

To give further clarification on python's `bytes` type, under the hood, it's literally just a contiguous array of 1 byte values, but with python magic wrapped over it. TLDR it behaves like `const uint8_t[]`. It's also handy because out of the box it provides slicing that's easy to use. An example of the slicing behaviour is in the code below.

I suppose we could theoretically use `str` or `list` here too but that would require extra work. 

e.g. using `str` forces me to encode to bytes before calculating exact bit lengths or applying masks anyway

e.g.2 using `list[int]` of raw byte values wastes memory, loses the super easy byte slicing provided by `bytes` and also require extra conversion steps during the block processing, which is really unnecessary

In [3]:
message = "abc"
message_bytes = message.encode("utf-8")

print("message_bytes = ", message_bytes)
print("len(message_bytes) = ", len(message_bytes))

# A bytes slice keeps the same bytes type and returns the selected range.
block = b"\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0A"
chunk = block[0:4]
print("chunk = ", chunk)

message_bytes =  b'abc'
len(message_bytes) =  3
chunk =  b'\x01\x02\x03\x04'


### 512-bit message blocks
Once the input message has been padded, sha256 processes it in 512bit chunks (64 bytes) AKA message blocks. Since the input msg is represented in `bytes`, naturally, each individual block would be a subslice of that big padded input message, meaning they would all be bytes that are 64-byte wide (let's ignore the metadata python attaches to the `bytes` object). And obviously, if there's more than 1 block, I'll be using a standard list ie `list[bytes]`

In [4]:
block = bytes(range(64))

assert len(block) == 64
assert len(block) * 8 == 512

# SHA-256 padding ends with the original bit length encoded as a big-endian
# 64-bit value; the preceding bytes contain the 1-bit marker and zero fill.
padded_msg = b"abc" + b"\x80" + b"\x00" * 52 + (24).to_bytes(8, "big")
assert len(padded_msg) == 64

blocks = [
    padded_msg[i : i + 64]
    for i in range(0, len(padded_msg), 64)
]
assert len(blocks) == 1
assert all(len(message_block) == 64 for message_block in blocks)

### the final 256-bit hash value
There's actually 2 useful representation for this, the final hash can be:
1. a 32 bytes for binary ops or storage
2. 64-char hexadecimal string for display (people usually display hashes in this because if we actually printed the 32 raw bytes, it'll be unreadable gibberish since many byte values won't correspond to printable ascii characters)

In [5]:
digest_bytes = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

assert len(digest_bytes) == 32
assert len(digest_bytes) * 8 == 256

print()
print("the raw bytes = ", digest_bytes)  # b'\xba\x78\x16\xbf... you get the idea, but remember the python magic i talked about above, so it might show some ascii when printed
digest_hex = digest_bytes.hex()
print("the hex display = ", digest_hex)  # ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


the raw bytes =  b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'
the hex display =  ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### big-endian integers
big-endian just means it stores the most significant byte first, e.g. the integer 0x12345678 is stored as:
```
12 34 56 78
```
The endianness matters at three places in SHA-256:
1. at the input boundary: when we split the 64 byte message block (512bit) into 16 initial 32-bit words ($W_0 \dots W_{15}$). The FIPS standard says to interpret each 4-byte group as a big-endian 32-bit integer, so that's what we're following
2. during padding: the original message length in bits is appended as a 64-bit big-endian integer
3. at the output: when we convert the final 32-bit state variables ($H_0 \dots H_7$) back into the final 32-byte hash array, once again just following FIPS, we write out each state integer as 4 big-endian bytes.

I'll show the input and output boundaries in the code below. The padding example above demonstrates the big-endian length field.

NOTE:
inside the main loop, the values are represented as `np.uint32` and the bit shifts, rotations, `add32`, `Ch`, and `Maj` operations do not involve endianness. Endianness only applies when bytes are interpreted as integers or integers are written back into a byte sequence. The conversion helpers handle those byte boundaries.

In [6]:
value = 0x12345678

encoded = value.to_bytes(4, byteorder="big")
decoded = int.from_bytes(encoded, byteorder="big")

print(encoded)  # b'\x124Vx'
print(decoded == value)  # True

b'\x124Vx'
True


### implementation of data types
Below is the setup for the data types used by future problems

In [7]:
import numpy as np

UINT32_MASK = 0xFFFFFFFF


def to_uint32(value: int | np.integer) -> np.uint32:
    """Convert an integer to a wrapped 32-bit unsigned NumPy value."""
    # Convert NumPy scalar types before applying the Python integer mask.
    return np.uint32(int(value) & UINT32_MASK)


def to_uint32_array(values: list[int] | tuple[int, ...]) -> np.ndarray:
    """Convert a sequence of integers into a mutable uint32 NumPy array."""
    return np.array([to_uint32(value) for value in values], dtype=np.uint32)


def block_to_words(block: bytes) -> np.ndarray:
    """Decode one 64-byte, big-endian SHA-256 block into sixteen uint32 words."""
    if len(block) != 64:
        raise ValueError("a SHA-256 block must contain exactly 64 bytes")

    # >u4 means big-endian, unsigned four-byte integers. astype then converts
    # them to native uint32 values for the bitwise operations that follow.
    return np.frombuffer(block, dtype=">u4").astype(np.uint32)


def words_to_bytes(words: list[int] | tuple[int, ...] | np.ndarray) -> bytes:
    """Encode uint32 words as a big-endian byte sequence."""
    uint32_words = to_uint32_array(list(words))
    return uint32_words.astype(">u4").tobytes()


word = to_uint32(0x12345678)
words = to_uint32_array([0x12345678, 0x90ABCDEF, 0x00000001])
block = bytes(range(64))
block_words = block_to_words(block)

assert isinstance(word, np.uint32)
assert words.dtype == np.uint32
assert len(block_words) == 16
assert all(isinstance(value, np.uint32) for value in block_words)
assert words_to_bytes([0x12345678]) == bytes.fromhex("12345678")

print(f"word = {word:08x} ({type(word).__name__})")
print(f"words dtype = {words.dtype}")
print(f"block contains {len(block_words)} uint32 words")

word = 12345678 (uint32)
words dtype = uint32
block contains 16 uint32 words


### Tests for the data type helpers (just in case)
The examples above demonstrate typical usage. The tests below verify the required types, 32-bit wrapping, mutability, big-endian conversion, validation, and round-trip behavior. I know I already added assertions above but this will might save me some headache down the line

In [8]:
import unittest

import numpy as np


class TestSha256DataTypes(unittest.TestCase):
    def test_to_uint32_returns_uint32_and_wraps_values(self):
        self.assertIsInstance(to_uint32(0x12345678), np.uint32)
        self.assertEqual(int(to_uint32(0x1_0000_0001)), 0x00000001)
        # -1 becomes all 32 bits set after masking.
        self.assertEqual(int(to_uint32(-1)), UINT32_MASK)

    def test_to_uint32_array_returns_mutable_uint32_array(self):
        values = to_uint32_array([0x12345678, 0x90ABCDEF])

        self.assertIsInstance(values, np.ndarray)
        self.assertEqual(values.dtype, np.dtype(np.uint32))
        self.assertEqual(values.tolist(), [0x12345678, 0x90ABCDEF])

        values[0] = to_uint32(0xCAFEBABE)
        self.assertEqual(int(values[0]), 0xCAFEBABE)

    def test_block_to_words_decodes_big_endian_words(self):
        block = bytes(range(64))
        words = block_to_words(block)

        self.assertEqual(words.dtype, np.dtype(np.uint32))
        self.assertEqual(len(words), 16)
        self.assertEqual(int(words[0]), 0x00010203)
        self.assertEqual(int(words[-1]), 0x3C3D3E3F)

    def test_block_to_words_rejects_invalid_block_length(self):
        with self.assertRaises(ValueError):
            block_to_words(b"too short")

    def test_words_to_bytes_encodes_big_endian_words(self):
        words = [0x12345678, 0x90ABCDEF]
        self.assertEqual(
            words_to_bytes(words),
            bytes.fromhex("1234567890abcdef"),
        )

    def test_block_word_conversion_round_trips(self):
        block = bytes(range(64))
        self.assertEqual(words_to_bytes(block_to_words(block)), block)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSha256DataTypes)
)
assert result.wasSuccessful()

test_block_to_words_decodes_big_endian_words (__main__.TestSha256DataTypes) ... ok
test_block_to_words_rejects_invalid_block_length (__main__.TestSha256DataTypes) ... ok
test_block_word_conversion_round_trips (__main__.TestSha256DataTypes) ... ok
test_to_uint32_array_returns_mutable_uint32_array (__main__.TestSha256DataTypes) ... ok
test_to_uint32_returns_uint32_and_wraps_values (__main__.TestSha256DataTypes) ... ok
test_words_to_bytes_encodes_big_endian_words (__main__.TestSha256DataTypes) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.012s

OK


## Problem 2: SHA-256 Bitwise Operations

Before we go into actually implementing the functions, let's go through a big picture explanation first, so we can see where the functions we are going to implement like Ch() and Maj() comes into play in the big picture:

SHA-256 takes input messages and turns them into a fixed 256-bit signature through iterative mixing. To make the output completely unpredictable and impossible to reverse, sha256 breaks every message into 512bit blocks, expands each block into a 64-word schedule, and runs 64 compression rounds over eight state variables ($a, b, c, d, e, f, g, h$).

The six functions in Problem 2's task are the elementary mixing building blocks used inside those 64 rounds:
- The Sigmas ($\sigma_0, \sigma_1, \Sigma_0, \Sigma_1$) act as diffusion engines. They take a word, rotate its bits by different distances, and XOR them together. This ensures that altering even a single bit in your input completely scrambles every bit in future steps AKA the avalanche effect.
- Ch and Maj act as non-linear logic gates. They combine bits from three different state variables so that an attacker cannot solve for past inputs using simple linear algebra. (kinda like how activation functions in ML NN stops the whole network from collapsing into a single linear alg equation, they do the same job here, provide non linearity)

To show you where they fit in, you can refer to my simplified high level idea of the whole process as a graph below:
```
[ Input Message ] 
       │
       ▼
 1. Preprocessing (Padding + Length Attachment)
       │
       ▼
 2. Break into 512-bit Blocks (M¹, M², ..., Mⁿ)
       │
       ▼
 3. Compression Loop (For each 512-bit block):
       ├── Message Schedule Expansion (64 words: W₀ ... W₆₃)  <-- Uses sigma0, sigma1
       └── 64 Rounds of Transformation (Working vars A..H)   <-- Uses Ch, Maj, Sigma0, Sigma1
       │
       ▼
 4. Final Hash Output (Concatenate H₀ ... H₇ into 256 bits)
```

### explanation of the Message Schedule Expansion before we even do the implementation
easier to show with a graph for what the MSE means:
```
Original 512-bit Block (16 words of 32 bits each):
[ W₀ ][ W₁ ][ W₂ ][ W₃ ] ... [ W₁₄ ][ W₁₅ ]
  │     │     │     │           │      │
  ▼     ▼     ▼     ▼           ▼      ▼
[ W₀ ][ W₁ ][ W₂ ][ W₃ ] ... [ W₁₄ ][ W₁₅ ]  <-- First 16 words are copied directly
                                   │
                     ┌─────────────┴─────────────┐
                     │ Generated mathematically! │
                     ▼                           ▼
            [ W₁₆ ][ W₁₇ ] ... [ W₆₂ ][ W₆₃ ] <-- 48 brand-new 32-bit words created
```

What I mean by generated mathematically:
$$W_t = \sigma^1_{256}(W_{t-2}) + W_{t-7} + \sigma^0_{256}(W_{t-15}) + W_{t-16} \pmod{2^{32}}$$

e.g. to generate the 17th and 18th word ($W_{16}$ and $W_{17}$)
- $W_{16} = \sigma_1(W_{14}) + W_9 + \sigma_0(W_1) + W_0 \pmod{2^{32}} \text{  (New 32-bit Word) }$
- $W_{17} = \sigma_1(W_{15}) + W_10 + \sigma_0(W_2) + W_1 \pmod{2^{32}}$
- so on and so forth until $W_{63}$, which is the last, 64th word

### the 64 rounds of transformation/scrambling
so after we get the 64 words from the step before, we do 64 rounds, for each round we get the t-th Word ($W_t$) and Constant ($K_t$) and we have a bunch of working variables `a` to `h`, they basically just preserve the state from one round to the next, think of it like the KV cache of each LLM decode step.

NOTE: the initial working var for round 1 and also the constants K0 to K63 is generated using maths, we will tackle this in Problem 3 so we will ignore that for now
```
Working Variables    (a,  b,  c,  d,  e,  f,  g,  h)
                      │   │   │       │   │   │   │
                      ▼   ▼   ▼       ▼   ▼   ▼   │
                    ┌───────────┐   ┌───────────┐ │
                    │ Sigma0(a) │   │ Sigma1(e) │ │
                    └─────┬─────┘   └─────┬─────┘ │
                          │               │       │
                    ┌─────┴─────┐   ┌─────┴─────┐ │
                    │ Maj(a,b,c)│   │  Ch(e,f,g)│ │
                    └─────┬─────┘   └─────┬─────┘ │
                          │               │       │
                          ▼               ▼       ▼
                       [  T₂  ]        [     T₁    ]  <── (also mixes in W_t and K_t)
                          │               │
                          └───────┬───────┘
                                  ▼
                  Update variables (a,b,c,d,e,f,g,h)
                      for the NEXT round (t + 1)
```

### `Ch(x,y,z)`
Ch means Choose. For each bit:
If the corresponding bit in x is 1, choose the bit from y.
If the corresponding bit in x is 0, choose the bit from z.

It's easier to see with an example so that's what I'm going to be showing for each function.

#### Example
Using 4-bit values:
```text
x = 1010
y = 1100
z = 0011
```
Calculate:
```text
x AND y       = 1000
NOT x AND z   = 0101 AND 0011 = 0001
1000 XOR 0001 = 1001
```
Therefore:
```text
Ch(1010, 1100, 0011) = 1001
```
The result selects bits from y where x has `1`s, and bits from z where x has `0`s, that's all there is to it

#### implementation for `Ch(x, y, z)`

In [9]:
def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Choose bits from y or z according to the corresponding bits in x."""
    x_value = int(x)
    y_value = int(y)
    z_value = int(z)

    # Python's ~ is unbounded, so mask its result back to one word before
    # using it to select bits from z.
    selected_y_bits = x_value & y_value
    selected_z_bits = (~x_value & UINT32_MASK) & z_value

    return to_uint32(selected_y_bits ^ selected_z_bits)

#### unit test for `Ch(x, y, z)`

In [10]:
import unittest

import numpy as np


class TestCh(unittest.TestCase):
    def test_ch_selects_y_when_x_bit_is_one_and_z_when_x_bit_is_zero(self):
        x = np.uint32(0b1010)
        y = np.uint32(0b1100)
        z = np.uint32(0b0011)
        expected = np.uint32(0b1001)
        actual = Ch(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_ch_returns_y_when_x_contains_only_ones(self):
        x = np.uint32(0xFFFFFFFF)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)
        actual = Ch(x, y, z)

        self.assertEqual(actual, y)
        self.assertIsInstance(actual, np.uint32)

    def test_ch_returns_z_when_x_contains_only_zeros(self):
        x = np.uint32(0x00000000)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)
        actual = Ch(x, y, z)

        self.assertEqual(actual, z)
        self.assertIsInstance(actual, np.uint32)

    def test_ch_uses_the_sha256_bitwise_formula(self):
        x = np.uint32(0xAAAAAAAA)
        y = np.uint32(0xCCCCCCCC)
        z = np.uint32(0x33333333)
        expected = np.uint32(
            (int(x) & int(y)) ^ ((~int(x) & UINT32_MASK) & int(z))
        )
        actual = Ch(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestCh)
)
assert result.wasSuccessful()

test_ch_returns_y_when_x_contains_only_ones (__main__.TestCh) ... ok
test_ch_returns_z_when_x_contains_only_zeros (__main__.TestCh) ... ok
test_ch_selects_y_when_x_bit_is_one_and_z_when_x_bit_is_zero (__main__.TestCh) ... ok
test_ch_uses_the_sha256_bitwise_formula (__main__.TestCh) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK


### `Maj(x, y, z)`

Maj means majority. For each bit position, it returns the bit that appears most often among x,y,z, it's a pretty straight forward one.

#### Example
Using 4-bit values (same as Ch for simplicity sake):
```text
x = 1010
y = 1100
z = 0011
```
Calculate:
```text
x AND y = 1000
x AND z = 0010
y AND z = 0000

then 

1000 XOR 0010 XOR 0000 = 1010
```
Therefore:
```text
Maj(1010, 1100, 0011) = 1010
```
Basically, each res bit is just the one that appears the most between x y and z. 

#### Implementation for `Maj(x, y, z)`

In [11]:
def Maj(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the bitwise majority value of three uint32 words."""
    x_value = int(x)
    y_value = int(y)
    z_value = int(z)

    pairwise_majority = (
        (x_value & y_value)
        ^ (x_value & z_value)
        ^ (y_value & z_value)
    )

    return to_uint32(pairwise_majority)

#### Unit test for `Maj(x, y, z)`

In [12]:
import unittest

import numpy as np


class TestMaj(unittest.TestCase):
    def test_maj_returns_the_bitwise_majority(self):
        x = np.uint32(0b1010)
        y = np.uint32(0b1100)
        z = np.uint32(0b0011)
        expected = np.uint32(0b1010)
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_with_all_one_x_selects_the_union_of_y_and_z(self):
        x = np.uint32(0xFFFFFFFF)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)
        expected = np.uint32(int(y) | int(z))
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_with_all_zero_x_selects_the_intersection_of_y_and_z(self):
        x = np.uint32(0x00000000)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)
        expected = np.uint32(int(y) & int(z))
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_matches_the_sha256_bitwise_formula(self):
        x = np.uint32(0xAAAAAAAA)
        y = np.uint32(0xCCCCCCCC)
        z = np.uint32(0x33333333)
        expected = np.uint32(
            (int(x) & int(y)) ^ (int(x) & int(z)) ^ (int(y) & int(z))
        )
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestMaj)
)
assert result.wasSuccessful()

test_maj_matches_the_sha256_bitwise_formula (__main__.TestMaj) ... ok
test_maj_returns_the_bitwise_majority (__main__.TestMaj) ... ok
test_maj_with_all_one_x_selects_the_union_of_y_and_z (__main__.TestMaj) ... ok
test_maj_with_all_zero_x_selects_the_intersection_of_y_and_z (__main__.TestMaj) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


### `Sigma0(x)` $\Sigma^0_{256}(x)$

`Sigma0` is one of the two uppercase Sigma functions used during the SHA-256 compression rounds. It mixes the working variable `a` before that value is combined with `Maj(a, b, c)`.

keep in mind that `a`, `b`, and `c` are three of SHA-256's eight current working variables. Each working variable is a 32-bit word that stores part of the intermediate hash state.

In fancy math terms, this Sigma0 function is defined as:

$$\Sigma^0_{256}(x) = ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$$

To put it simply, take the input word `x`, make three rotated copies of it, and combine those copies with XOR. `ROTR` means rotate right. Unlike a normal right shift, the bits that leave the right-hand side are wrapped around to the left, so no information is discarded. The three rotations spread each input bit to different output positions, then XOR combines all three results bit by bit. $ROTR^{n}(x)$ just means by n position

e.g. 1234, i know this isn't binary but it's clearer. 
- $ROTR^{1}(1234) = 4123$
- $ROTR^{2}(1234) = 3412$
- $ROTR^{3}(1234) = 2341$ you get the point

#### Example
```text
x = 10110001
```
calculate
```text
ROTR^2(x) = 01101100
```
The two `1` bits that fell off the right side, `01`, wrapped around to the left. The real SHA-256 function does this with a 32-bit word and uses three rotations:

```text
ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x)
```

The final result is another 32-bit word. Don't think of the result as a readable number, the important point is that one input word has been rearranged in three different ways and then mixed together bit by bit.

#### Implementation for `Sigma0(x)` $\Sigma^0_{256}(x)$

In [13]:
def Sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma0 transformation."""
    x_value = int(x)

    # Each rotate-right expression shifts bits right and brings the shifted-out
    # bits back on the left; the mask keeps the intermediate word bounded.
    rotate_right_2 = ((x_value >> 2) | (x_value << 30)) & UINT32_MASK
    rotate_right_13 = ((x_value >> 13) | (x_value << 19)) & UINT32_MASK
    rotate_right_22 = ((x_value >> 22) | (x_value << 10)) & UINT32_MASK

    return to_uint32(rotate_right_2 ^ rotate_right_13 ^ rotate_right_22)

#### Unit test for `Sigma0(x)` $\Sigma^0_{256}(x)$

In [14]:
import unittest

import numpy as np


class TestSigma0(unittest.TestCase):
    def test_sigma0_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0x66146474),
            (0x00000001, 0x40080400),
            (0x80000000, 0x20040200),
            (0xFFFFFFFF, 0xFFFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = Sigma0(np.uint32(input_value))
                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSigma0)
)
assert result.wasSuccessful()

test_sigma0_matches_sha256_reference_vectors (__main__.TestSigma0) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


### `Sigma1(x)` $\Sigma^1_{256}(x)$

`Sigma1` is **literally** the same as `Sigma0` the only 2 difference is that it uses different n during ROTR, ie it is defined as:
$$\Sigma^1_{256}(x) = ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$$

second difference is that it is applied to working var `e`, recall that the abcdefg are the eight 32-bit working variables used by SHA-256 during its compression rounds. Therefore, when `Sigma1` is applied to `e`, the relevant working variables around it are `e`, `f`, and `g`. These variables are later used by the `Ch(e, f, g)` function.

So, in simple terms, `Sigma1` is the same type of bit-mixing operation as `Sigma0`, but it uses different rotation distances and operates on a different working variable.

No example, it's the same as before

#### Implementation for `Sigma1(x)` $\Sigma^1_{256}(x)$

In [15]:
def Sigma1(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma1 transformation."""
    x_value = int(x)

    # Sigma1 uses the same rotate construction as Sigma0, with its own
    # distances specified by SHA-256.
    rotate_right_6 = ((x_value >> 6) | (x_value << 26)) & UINT32_MASK
    rotate_right_11 = ((x_value >> 11) | (x_value << 21)) & UINT32_MASK
    rotate_right_25 = ((x_value >> 25) | (x_value << 7)) & UINT32_MASK

    return to_uint32(rotate_right_6 ^ rotate_right_11 ^ rotate_right_25)

#### Unit test for `Sigma1(x)` $\Sigma^1_{256}(x)$

In [16]:
import unittest

import numpy as np


class TestSigma1(unittest.TestCase):
    def test_sigma1_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0x3561ABDA),
            (0x00000001, 0x04200080),
            (0x80000000, 0x02100040),
            (0xFFFFFFFF, 0xFFFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = Sigma1(np.uint32(input_value))
                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSigma1)
)
assert result.wasSuccessful()

test_sigma1_matches_sha256_reference_vectors (__main__.TestSigma1) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


### `sigma0(x)` $\sigma^0_{256}(x)$

`sigma0` is a lowercase Sigma function used while expanding the message schedule (mentioned in the initial explanation before we implement the 6 functions). It helps calculate later schedule words from words that appeared earlier, rather than mixing the eight working variables during a compression round. the 2 sigma function is part of this word expansion equation thing (recall the earlier graph)

$$W_t = \sigma^1_{256}(W_{t-2}) + W_{t-7} + \sigma^0_{256}(W_{t-15}) + W_{t-16} \pmod{2^{32}}$$

It is defined as:

$$\sigma^0_{256}(x) = ROTR^7(x) \oplus ROTR^{18}(x) \oplus SHR^3(x)$$

I already explained `ROTR` above so I'll skip that, but the `SHR` here means a logical right shift. It discards the bits that leave the right-hand side and fills the new left-hand positions with zeroes. That deliberate information loss makes the lowercase schedule functions different from the uppercase round functions, basically just the C/C++ `>>`

#### Example
```text
x = 10110001
ROTR^2(x) = 01101100
SHR^3(x)  = 00010110
```

The actual SHA-256 function uses `ROTR^7`, `ROTR^18`, and `SHR^3`. The 3 results are XORed together to produce the schedule-mixing value, it's just a different flavour of Sigma essentially.

#### Implementation for `sigma0(x)` $\sigma^0_{256}(x)$

In [17]:
def sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 lowercase sigma0 transformation."""
    x_value = int(x)

    rotate_right_7 = ((x_value >> 7) | (x_value << 25)) & UINT32_MASK
    rotate_right_18 = ((x_value >> 18) | (x_value << 14)) & UINT32_MASK
    # SHR discards shifted-out bits instead of wrapping them around.
    shift_right_3 = x_value >> 3

    return to_uint32(rotate_right_7 ^ rotate_right_18 ^ shift_right_3)

#### Unit test for `sigma0(x)` $\sigma^0_{256}(x)$

In [18]:
import unittest

import numpy as np


class Testsigma0(unittest.TestCase):
    def test_sigma0_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0xE7FCE6EE),
            (0x00000001, 0x02004000),
            (0x80000000, 0x11002000),
            (0xFFFFFFFF, 0x1FFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = sigma0(np.uint32(input_value))
                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Testsigma0)
)
assert result.wasSuccessful()

test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


### `sigma1(x)` $\sigma^1_{256}(x)$

`sigma1` is basically the same, only difference is it uses different n for $ROTR^{n}(x)$ and also $SHR^{n}(x)$, ie:

$$\sigma^1_{256}(x) = ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$$

Won't even give an example, you already get the idea from before

#### Implementation for `sigma1(x)` $\sigma^1_{256}(x)$

In [19]:
def sigma1(x: np.uint32) -> np.uint32:
    """Return the SHA-256 lowercase sigma1 transformation."""
    x_value = int(x)

    rotate_right_17 = ((x_value >> 17) | (x_value << 15)) & UINT32_MASK
    rotate_right_19 = ((x_value >> 19) | (x_value << 13)) & UINT32_MASK
    # As in sigma0, this SHR term is a shift rather than a rotation.
    shift_right_10 = x_value >> 10

    return to_uint32(rotate_right_17 ^ rotate_right_19 ^ shift_right_10)

#### Unit test for `sigma1(x)` $\sigma^1_{256}(x)$

In [20]:
import unittest

import numpy as np


class Testsigma1(unittest.TestCase):
    def test_sigma1_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0xA1F78649),
            (0x00000001, 0x0000A000),
            (0x80000000, 0x00205000),
            (0xFFFFFFFF, 0x003FFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = sigma1(np.uint32(input_value))
                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Testsigma1)
)
assert result.wasSuccessful()

test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


## Problem 3: Generating the SHA-256 Constants
Recall the graph that I showed for the 64 rounds of scrambling:
```
Working Variables    (a,  b,  c,  d,  e,  f,  g,  h)
                      │   │   │       │   │   │   │
                      ▼   ▼   ▼       ▼   ▼   ▼   │
                    ┌───────────┐   ┌───────────┐ │
                    │ Sigma0(a) │   │ Sigma1(e) │ │
                    └─────┬─────┘   └─────┬─────┘ │
                          │               │       │
                    ┌─────┴─────┐   ┌─────┴─────┐ │
                    │ Maj(a,b,c)│   │  Ch(e,f,g)│ │
                    └─────┬─────┘   └─────┬─────┘ │
                          │               │       │
                          ▼               ▼       ▼
                       [  T₂  ]        [     T₁    ]  <── (also mixes in W_t and K_t)
                          │               │
                          └───────┬───────┘
                                  ▼
                  Update variables (a,b,c,d,e,f,g,h)
                      for the NEXT round (t + 1)
```

so now we need to generate the 64 constants $K_0$ until $K_63$ as well as the init value for a,b,c...,h. Technically speaking, the variables are referred to as ($H_0 \dots H_7$) in math.
The init 8 values we're trying to gen are: ($H_0^{(0)} \dots H_7^{(0)}$).

These 8 values are the first 32 bits of the fractional parts of the square roots of the first 8 prime numbers ($2, 3, 5, 7, 11, 13, 17, 19$).
### Example: how to Calculate $H_0$ (Square Root of Prime 2)
1. Take prime $p = 2$
2. Compute $\sqrt{2} = 1.4142135623730951\dots$
3. Keep only the fractional part: $0.4142135623730951\dots$
4. Multiply by $2^{32}$ to extract the top 32 bits: $$0.4142135623730951 \times 2^{32} = 1779033703.95\dots$$
5. Convert the integer part ($1779033703$) to hexadecimal: $$\mathbf{\text{0x6a09e667}}$$

### same thing for $K_0 \dots K_{63}$
The $K$ constants use a nearly identical process, but with cube roots of the first 64 prime numbers ($2, 3, 5, \dots, 311$).
1. Take prime $p_t$ (e.g., $p_0 = 2$, $p_1 = 3$, $p_2 = 5$)
2. Compute $\sqrt[3]{p_t}$
3. Take the fractional part and multiply by $2^{32}$
4. Convert to hexadecimal
e.g. $K_0$ (from $\sqrt[3]{2}$): 0x428a2f98

In [21]:
# this is the first helper function we'll add 
# it'll be reused in both K constant and the 8 initial hash value H
def primes(n: int) -> list[int]:
    """Generate the first n prime numbers."""
    if n <= 0:
        return []

    primes_list = []
    candidate = 2  # Start checking for primes from the first prime number.

    while len(primes_list) < n:
        is_prime = True
        for p in primes_list:
            if p * p > candidate:  # No need to check beyond the square root.
                break
            if candidate % p == 0:
                is_prime = False
                break
        if is_prime:
            primes_list.append(candidate)
        candidate += 1

    return primes_list

# example of the first 8 primes printed out
print(primes(8))  # [2, 3, 5, 7, 11, 13, 17, 19]

[2, 3, 5, 7, 11, 13, 17, 19]


### implementation of the actual functions to get K constants and Hs

In [ ]:
# this helper is shared by both kinds of constants:
# H0..H7 use square roots, while K0..K63 use cube roots
def fractional_root_constants(prime_values: list[int], root: int) -> list[np.uint32]:
    """Convert fractional parts of prime roots into SHA-256 words."""
    constants = []

    for prime in prime_values:
        # Calculate the requested root for this prime, then remove its integer
        # part so only the fractional part remains.
        root_value = prime ** (1 / root)
        fractional_part = root_value - math.floor(root_value)

        scale = 1 << 32 # this is just 2^32, classic math trick m << n = m * 2^n
        # we multiply by scale because SHA wants to store everything in uint32 words 
        # 0 <= fractional_part <= 1, so multiplying by 2^32 gives us a number in the range [0, 2^32)
        # basically we express the fractional part as an uint WHILE still keeping the same ratio
        # ie 0 maps to 0, 1 maps to 2^32 - 1
        constants.append(to_uint32(int(fractional_part * (scale))))

    return constants


# The initial hash state uses the fractional parts of the square roots of
# the first eight primes: 2, 3, 5, 7, 11, 13, 17, and 19.
def generate_initial_hash_values() -> list[np.uint32]:
    """Generate H0 through H7 from square roots of the first eight primes."""
    return fractional_root_constants(primes(8), root=2)


# The round constants use the fractional parts of the cube roots of the
# first 64 primes, one constant for each compression round.
def generate_round_constants() -> list[np.uint32]:
    """Generate K0 through K63 from cube roots of the first 64 primes."""
    return fractional_root_constants(primes(64), root=3)

### Unit test for $H_0^{(0)} \dots H_7^{(0)}$ and $K_0 \dots K_{63}$

In [29]:
import math
import unittest

import numpy as np


# values published in FIPS 180-4 and used as the test oracle. got them straight from the wiki page
EXPECTED_INITIAL_HASH_VALUES = [
    0x6A09E667,
    0xBB67AE85,
    0x3C6EF372,
    0xA54FF53A,
    0x510E527F,
    0x9B05688C,
    0x1F83D9AB,
    0x5BE0CD19,
]
# same for these
EXPECTED_ROUND_CONSTANTS = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5,
    0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3,
    0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC,
    0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7,
    0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13,
    0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3,
    0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5,
    0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208,
    0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

class TestSha256Constants(unittest.TestCase):
    def test_primes_produces_the_first_64_primes(self):
        generated_primes = primes(64)

        self.assertEqual(generated_primes[:8], [2, 3, 5, 7, 11, 13, 17, 19])
        self.assertEqual(len(generated_primes), 64)
        self.assertEqual(len(set(generated_primes)), 64)

    def test_fractional_cube_root_conversion_produces_k0(self):
        cube_root = 2 ** (1 / 3)
        fractional_part = cube_root - math.floor(cube_root)
        generated = fractional_root_constants([2], root=3)[0]

        self.assertGreater(fractional_part, 0)
        self.assertLess(fractional_part, 1)
        self.assertEqual(generated, np.uint32(EXPECTED_ROUND_CONSTANTS[0]))
        self.assertIsInstance(generated, np.uint32)

    def test_initial_hash_values_match_the_secure_hash_standard(self):
        actual = generate_initial_hash_values()
        expected = [np.uint32(value) for value in EXPECTED_INITIAL_HASH_VALUES]

        self.assertEqual(actual, expected)

    def test_round_constants_match_the_secure_hash_standard(self):
        actual = generate_round_constants()
        expected = [np.uint32(value) for value in EXPECTED_ROUND_CONSTANTS]

        self.assertEqual(actual, expected)
        self.assertEqual(len(actual), 64)
        self.assertTrue(all(isinstance(value, np.uint32) for value in actual))


initial_hash_values = generate_initial_hash_values()
round_constants = generate_round_constants()

print("H constants:")
print(" ".join(f"0x{int(value):08x}" for value in initial_hash_values))
print("K constants:")
print(" ".join(f"0x{int(value):08x}" for value in round_constants))

result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSha256Constants)
)
assert result.wasSuccessful()

test_fractional_cube_root_conversion_produces_k0 (__main__.TestSha256Constants) ... ok
test_initial_hash_values_match_the_secure_hash_standard (__main__.TestSha256Constants) ... ok
test_primes_produces_the_first_64_primes (__main__.TestSha256Constants) ... ok
test_round_constants_match_the_secure_hash_standard (__main__.TestSha256Constants) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.006s

OK


H constants:
0x6a09e667 0xbb67ae85 0x3c6ef372 0xa54ff53a 0x510e527f 0x9b05688c 0x1f83d9ab 0x5be0cd19
K constants:
0x428a2f98 0x71374491 0xb5c0fbcf 0xe9b5dba5 0x3956c25b 0x59f111f1 0x923f82a4 0xab1c5ed5 0xd807aa98 0x12835b01 0x243185be 0x550c7dc3 0x72be5d74 0x80deb1fe 0x9bdc06a7 0xc19bf174 0xe49b69c1 0xefbe4786 0x0fc19dc6 0x240ca1cc 0x2de92c6f 0x4a7484aa 0x5cb0a9dc 0x76f988da 0x983e5152 0xa831c66d 0xb00327c8 0xbf597fc7 0xc6e00bf3 0xd5a79147 0x06ca6351 0x14292967 0x27b70a85 0x2e1b2138 0x4d2c6dfc 0x53380d13 0x650a7354 0x766a0abb 0x81c2c92e 0x92722c85 0xa2bfe8a1 0xa81a664b 0xc24b8b70 0xc76c51a3 0xd192e819 0xd6990624 0xf40e3585 0x106aa070 0x19a4c116 0x1e376c08 0x2748774c 0x34b0bcb5 0x391c0cb3 0x4ed8aa4a 0x5b9cca4f 0x682e6ff3 0x748f82ee 0x78a5636f 0x84c87814 0x8cc70208 0x90befffa 0xa4506ceb 0xbef9a3f7 0xc67178f2
